# 05 - Office Footfall Forecasting

## Objective

The objective of this notebook is to create a forecasting-ready campus attendance dataset and explore historical office footfall patterns that may support future attendance forecasting.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent

processed_path = project_root / "data" / "processed"

file_path = processed_path / "combined_no_exact_duplicates.csv"

df = pd.read_csv(
    file_path,
    low_memory=False
)

print(df.shape)

In [ ]:
workday_df = df[
    df["WEEKOFF_HOLIDAY"] == "WORKDAY"
].copy()

print(workday_df.shape)

In [ ]:
campus_workday_df = workday_df[
    workday_df["REMARKS"].isin(
        [
            "PRESENT",
            "Wipro Office",
            "TEMP CARD"
        ]
    )
].copy()

print(campus_workday_df.shape)

## Footfall Definition

For this notebook, daily campus footfall is defined as the number of unique employees who have at least one Campus / Office Presence record on a given date.

Since employees may have multiple project-level records on the same date, record count is not used as footfall. Instead, `EMP_CODE.nunique()` is used to count each employee only once per date.

In [ ]:
daily_footfall = (
    campus_workday_df
    .groupby("ATTENDANCE_DATE")["EMP_CODE"]
    .nunique()
    .reset_index(name="daily_campus_footfall")
)

daily_footfall["ATTENDANCE_DATE"] = pd.to_datetime(
    daily_footfall["ATTENDANCE_DATE"]
)

daily_footfall.head()

In [ ]:
print("Rows:", daily_footfall.shape[0])
print("Min Date:", daily_footfall["ATTENDANCE_DATE"].min())
print("Max Date:", daily_footfall["ATTENDANCE_DATE"].max())
print("Unique Dates:", daily_footfall["ATTENDANCE_DATE"].nunique())

In [ ]:
location_daily_footfall = (
    campus_workday_df
    .groupby(["ATTENDANCE_DATE", "LOCATION"])["EMP_CODE"]
    .nunique()
    .reset_index(name="location_daily_campus_footfall")
)

In [ ]:
date_range = pd.date_range(
    start=daily_footfall["ATTENDANCE_DATE"].min(),
    end=daily_footfall["ATTENDANCE_DATE"].max(),
    freq="D"
)

missing_dates = date_range.difference(daily_footfall["ATTENDANCE_DATE"])

print("Expected calendar dates:", len(date_range))
print("Available dates:", daily_footfall["ATTENDANCE_DATE"].nunique())
print("Missing dates:", len(missing_dates))

missing_dates

In [ ]:
daily_footfall["weekday_name"] = daily_footfall["ATTENDANCE_DATE"].dt.day_name()

daily_footfall.head()

In [ ]:
daily_footfall["daily_campus_footfall"].describe()

In [ ]:
daily_footfall["weekday_name"] = daily_footfall["ATTENDANCE_DATE"].dt.day_name()

daily_footfall["is_weekend"] = daily_footfall["weekday_name"].isin(
    ["Saturday", "Sunday"]
)

daily_footfall.groupby("is_weekend")["daily_campus_footfall"].describe()

In [ ]:
weekday_footfall_summary = (
    daily_footfall
    .groupby("weekday_name")["daily_campus_footfall"]
    .agg(
        average_footfall="mean",
        median_footfall="median",
        max_footfall="max",
        min_footfall="min",
        number_of_days="count"
    )
    .reset_index()
)

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"
]

weekday_footfall_summary["weekday_name"] = pd.Categorical(
    weekday_footfall_summary["weekday_name"],
    categories=weekday_order,
    ordered=True
)

weekday_footfall_summary = weekday_footfall_summary.sort_values("weekday_name")

weekday_footfall_summary

## Date Range and Footfall Series Validation

The daily campus footfall series covers 61 continuous calendar dates from 2025-04-01 to 2025-05-31.

There are no missing calendar dates in the daily footfall dataset, which makes the time series structurally complete for exploratory forecasting analysis.

However, the daily footfall distribution shows high variability. The mean footfall is lower than the median, suggesting that some low-footfall days, likely weekends or non-standard workdays, are pulling the average downward.

Therefore, weekday and weekend patterns should be analyzed separately before building any forecasting model.

## Employee Weekly Office-Day Combination Analysis

Since the weekday footfall analysis showed higher attendance on Tuesday and Wednesday and lower attendance on Friday, this section explores whether employees tend to cluster their office presence into specific weekday combinations.

The goal is to understand whether common hybrid work patterns exist, such as Monday-Tuesday-Wednesday or Tuesday-Wednesday-Thursday office attendance.

In [ ]:
# Create employee-date level campus presence data
campus_employee_day = (
    campus_workday_df[
        ["EMP_CODE", "ATTENDANCE_DATE"]
    ]
    .drop_duplicates()
    .copy()
)

campus_employee_day["ATTENDANCE_DATE"] = pd.to_datetime(
    campus_employee_day["ATTENDANCE_DATE"]
)

# Add weekday details
campus_employee_day["weekday_name"] = (
    campus_employee_day["ATTENDANCE_DATE"]
    .dt.day_name()
)

campus_employee_day["weekday_num"] = (
    campus_employee_day["ATTENDANCE_DATE"]
    .dt.weekday
)

# Keep only Monday-Friday for hybrid work pattern analysis
campus_employee_day_weekdays = campus_employee_day[
    campus_employee_day["weekday_num"] <= 4
].copy()

# Create week start date
campus_employee_day_weekdays["week_start"] = (
    campus_employee_day_weekdays["ATTENDANCE_DATE"]
    .dt.to_period("W-SUN")
    .apply(lambda x: x.start_time)
)

weekday_order = {
    "Monday": 1,
    "Tuesday": 2,
    "Wednesday": 3,
    "Thursday": 4,
    "Friday": 5
}

def ordered_weekday_combo(days):
    unique_days = sorted(
        set(days),
        key=lambda x: weekday_order[x]
    )
    return " + ".join(unique_days)

employee_weekly_office_pattern = (
    campus_employee_day_weekdays
    .groupby(["EMP_CODE", "week_start"])
    .agg(
        campus_days_in_week=("ATTENDANCE_DATE", "nunique"),
        weekday_combo=("weekday_name", ordered_weekday_combo)
    )
    .reset_index()
)

employee_weekly_office_pattern.head()

In [ ]:
three_day_office_patterns = employee_weekly_office_pattern[
    employee_weekly_office_pattern["campus_days_in_week"] == 3
].copy()

three_day_combo_summary = (
    three_day_office_patterns["weekday_combo"]
    .value_counts()
    .reset_index()
)

three_day_combo_summary.columns = [
    "weekday_combo",
    "employee_week_count"
]

three_day_combo_summary["employee_week_pct"] = (
    three_day_combo_summary["employee_week_count"] /
    three_day_combo_summary["employee_week_count"].sum() * 100
)

three_day_combo_summary.head(10)

## Employee Weekly Office-Day Combination Insight

The 3-day office attendance combination analysis shows that employees do not distribute office presence randomly across weekdays.

The most common 3-day office combination is Monday + Tuesday + Wednesday, representing approximately 28.70% of employee-weeks with exactly three campus days.

The second most common combination is Tuesday + Wednesday + Thursday, representing approximately 19.98%.

The third most common combination is Wednesday + Thursday + Friday, representing approximately 18.54%.

Together, the top three 3-day combinations account for approximately 67.22% of all employee-weeks where employees had exactly three campus days.

This suggests that employees tend to cluster office attendance into specific weekday blocks, especially early-week and mid-week combinations.

This pattern helps explain why Tuesday and Wednesday show higher average campus footfall, while Friday has comparatively lower average footfall.

These findings can support future footfall forecasting because weekday combinations and hybrid attendance patterns appear to influence daily campus attendance.

In [ ]:
import matplotlib.pyplot as plt

top_3day_patterns = three_day_combo_summary.head(10).copy()

plt.figure(figsize=(12, 6))

bars = plt.barh(
    top_3day_patterns["weekday_combo"],
    top_3day_patterns["employee_week_pct"]
)

plt.title("Top 3-Day Campus Attendance Combinations")
plt.xlabel("Share of 3-Day Employee-Weeks (%)")
plt.ylabel("Weekday Combination")

plt.gca().invert_yaxis()

for bar in bars:
    width = bar.get_width()
    plt.text(
        width,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}%",
        va="center"
    )

plt.tight_layout()
plt.show()


## 3-Day Office Combination Insight

The 3-day campus attendance combination analysis shows that employees do not distribute office presence randomly across weekdays.

The most common 3-day combination is Monday + Tuesday + Wednesday, representing approximately 28.70% of employee-weeks with exactly three campus days.

The second most common combination is Tuesday + Wednesday + Thursday, representing approximately 19.98%.

The third most common combination is Wednesday + Thursday + Friday, representing approximately 18.54%.

Together, the top three combinations account for approximately 67.22% of all 3-day employee-weeks.

This suggests that employees tend to cluster office attendance into specific weekday blocks, especially early-week and mid-week combinations.

This pattern helps explain why Tuesday and Wednesday show higher average campus footfall, while Friday has comparatively lower average footfall.

These findings can support future footfall forecasting because weekday combinations and hybrid attendance patterns appear to influence daily campus attendance.

In [ ]:
location_daily_footfall = (
    campus_workday_df
    .groupby(["ATTENDANCE_DATE", "LOCATION"])["EMP_CODE"]
    .nunique()
    .reset_index(name="location_daily_campus_footfall")
)

location_daily_footfall["ATTENDANCE_DATE"] = pd.to_datetime(
    location_daily_footfall["ATTENDANCE_DATE"]
)

location_daily_footfall.head()

In [ ]:
print("Rows:", location_daily_footfall.shape[0])
print("Unique Dates:", location_daily_footfall["ATTENDANCE_DATE"].nunique())
print("Unique Locations:", location_daily_footfall["LOCATION"].nunique())

location_daily_footfall.head()

In [ ]:
location_footfall_summary = (
    location_daily_footfall
    .groupby("LOCATION")["location_daily_campus_footfall"]
    .agg(
        average_daily_footfall="mean",
        median_daily_footfall="median",
        max_daily_footfall="max",
        min_daily_footfall="min",
        number_of_days="count"
    )
    .reset_index()
)

location_footfall_summary = location_footfall_summary.sort_values(
    "average_daily_footfall",
    ascending=False
)

location_footfall_summary.head(15)

In [ ]:
import matplotlib.pyplot as plt

top_locations_footfall = location_footfall_summary.head(10).copy()

plt.figure(figsize=(12, 6))

bars = plt.barh(
    top_locations_footfall["LOCATION"],
    top_locations_footfall["average_daily_footfall"]
)

plt.title("Top Locations by Average Daily Campus Footfall")
plt.xlabel("Average Daily Campus Footfall")
plt.ylabel("Location")

plt.gca().invert_yaxis()

for bar in bars:
    width = bar.get_width()
    plt.text(
        width,
        bar.get_y() + bar.get_height() / 2,
        f"{width:,.0f}",
        va="center"
    )

plt.tight_layout()
plt.show()

## Location-wise Daily Footfall Insight

The location-wise daily footfall analysis shows that campus attendance is highly concentrated across a few major locations.

Bengaluru has the highest average daily campus footfall, followed by Hyderabad, Pune, Chennai, Kolkata, and Mumbai.

Key observations:

- Bengaluru has the highest average daily footfall at approximately 7,608 unique employees.
- Hyderabad is the second-highest location with approximately 6,878 unique employees.
- Pune, Chennai, Kolkata, and Mumbai also contribute meaningfully to overall office footfall.
- Smaller locations show much lower average daily footfall, which may make forecasting less stable for those locations.

For future forecasting, it may be useful to first focus on major locations with sufficient daily volume and consistent data coverage before extending the model to smaller locations.

In [ ]:
location_daily_footfall["weekday_name"] = (
    location_daily_footfall["ATTENDANCE_DATE"]
    .dt.day_name()
)

location_weekday_footfall = (
    location_daily_footfall
    .groupby(["LOCATION", "weekday_name"])["location_daily_campus_footfall"]
    .agg(
        average_footfall="mean",
        median_footfall="median",
        max_footfall="max",
        min_footfall="min",
        number_of_days="count"
    )
    .reset_index()
)

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

location_weekday_footfall["weekday_name"] = pd.Categorical(
    location_weekday_footfall["weekday_name"],
    categories=weekday_order,
    ordered=True
)

location_weekday_footfall = location_weekday_footfall.sort_values(
    ["LOCATION", "weekday_name"]
)

location_weekday_footfall.head(20)

In [ ]:
top_10_locations = (
    location_footfall_summary
    .head(10)["LOCATION"]
    .tolist()
)

top_location_weekday_footfall = location_weekday_footfall[
    location_weekday_footfall["LOCATION"].isin(top_10_locations)
].copy()

top_location_weekday_footfall.head(30)

In [ ]:
top_location_weekday_pivot = (
    top_location_weekday_footfall
    .pivot_table(
        index="LOCATION",
        columns="weekday_name",
        values="average_footfall",
        fill_value=0
    )
    .reset_index()
)

top_location_weekday_pivot

## Location-wise Weekday Footfall Insight

The location-wise weekday footfall analysis shows that most major locations follow a similar weekly attendance pattern.

Across major locations such as Bengaluru, Hyderabad, Pune, Chennai, Kolkata, Mumbai, and Coimbatore, Wednesday generally has the highest average campus footfall, followed closely by Tuesday.

Friday shows comparatively lower average footfall across most major locations, while Saturday and Sunday show very low campus attendance compared with regular weekdays.

This confirms that weekday effects are not only visible at the organization-wide level, but also at the location level.

For future location-wise footfall forecasting, weekday information should be included as an important predictive feature. Location-specific models or features may also be useful because each location has different baseline footfall volume.

In [ ]:
forecasting_df = location_daily_footfall.copy()

forecasting_df["ATTENDANCE_DATE"] = pd.to_datetime(
    forecasting_df["ATTENDANCE_DATE"]
)

forecasting_df["weekday_name"] = forecasting_df["ATTENDANCE_DATE"].dt.day_name()
forecasting_df["weekday_num"] = forecasting_df["ATTENDANCE_DATE"].dt.weekday
forecasting_df["is_weekend"] = forecasting_df["weekday_num"].isin([5, 6])

forecasting_df["month"] = forecasting_df["ATTENDANCE_DATE"].dt.month
forecasting_df["day"] = forecasting_df["ATTENDANCE_DATE"].dt.day
forecasting_df["week_of_year"] = forecasting_df["ATTENDANCE_DATE"].dt.isocalendar().week.astype(int)

forecasting_df.head()

In [ ]:
print("Forecasting dataset shape:", forecasting_df.shape)
print("Unique dates:", forecasting_df["ATTENDANCE_DATE"].nunique())
print("Unique locations:", forecasting_df["LOCATION"].nunique())

forecasting_df.head()

In [ ]:
major_forecast_locations = (
    location_footfall_summary
    .sort_values("average_daily_footfall", ascending=False)
    .head(10)["LOCATION"]
    .tolist()
)

major_forecast_locations

In [ ]:
major_location_forecasting_df = forecasting_df[
    forecasting_df["LOCATION"].isin(major_forecast_locations)
].copy()

print("Major location forecasting shape:", major_location_forecasting_df.shape)
print("Unique dates:", major_location_forecasting_df["ATTENDANCE_DATE"].nunique())
print("Unique locations:", major_location_forecasting_df["LOCATION"].nunique())

major_location_forecasting_df.head()

## Forecasting-ready Dataset

A location-wise daily campus footfall dataset was created for future forecasting.

Each row represents the number of unique employees with campus/office presence for a given date and location.

Target variable:

- location_daily_campus_footfall

Initial forecasting features:

- LOCATION
- weekday_name
- weekday_num
- is_weekend
- month
- day
- week_of_year

The dataset contains 61 calendar dates and 60 locations. For initial forecasting, it may be more reliable to focus on major locations with higher and more stable daily footfall volumes.

## Baseline Footfall Forecast using Location and Weekday Average

Before building any complex forecasting model, a simple baseline forecast is created.

The baseline forecast predicts expected campus footfall using the historical average footfall for each LOCATION and weekday combination.

For example, the expected Bengaluru Wednesday footfall is calculated as the historical average Bengaluru Wednesday campus footfall.

This baseline provides a simple and explainable benchmark for future forecasting models.

In [ ]:
location_weekday_baseline = (
    major_location_forecasting_df
    .groupby(["LOCATION", "weekday_name"])["location_daily_campus_footfall"]
    .mean()
    .reset_index(name="baseline_expected_footfall")
)

location_weekday_baseline.head(20)

In [ ]:
baseline_comparison_df = major_location_forecasting_df.merge(
    location_weekday_baseline,
    on=["LOCATION", "weekday_name"],
    how="left"
)

baseline_comparison_df.head()


In [ ]:
baseline_comparison_df["forecast_error"] = (
    baseline_comparison_df["location_daily_campus_footfall"] -
    baseline_comparison_df["baseline_expected_footfall"]
)

baseline_comparison_df["absolute_error"] = (
    baseline_comparison_df["forecast_error"].abs()
)

baseline_comparison_df.head()

In [ ]:
baseline_evaluation = (
    baseline_comparison_df
    .groupby("LOCATION")
    .agg(
        average_actual_footfall=("location_daily_campus_footfall", "mean"),
        average_baseline_footfall=("baseline_expected_footfall", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        max_absolute_error=("absolute_error", "max")
    )
    .reset_index()
)

baseline_evaluation = baseline_evaluation.sort_values(
    "average_actual_footfall",
    ascending=False
)

baseline_evaluation

## Baseline Forecast Interpretation

A simple location-weekday baseline forecast was created using historical average campus footfall for each LOCATION and weekday combination.

This baseline is explainable and gives an initial estimate of expected footfall for a given location and weekday.

However, this first evaluation is in-sample because the same historical period was used to calculate the baseline and compare actual values. Therefore, the equal average actual and average baseline values are expected.

The next step is to perform a time-based train-test split, where earlier dates are used to calculate the baseline and later dates are used for validation. This will provide a more realistic estimate of forecasting performance.

In [ ]:
# Sort data by date
major_location_forecasting_df = major_location_forecasting_df.sort_values(
    "ATTENDANCE_DATE"
).copy()

# Define time-based split
split_date = pd.to_datetime("2025-05-18")

train_df = major_location_forecasting_df[
    major_location_forecasting_df["ATTENDANCE_DATE"] < split_date
].copy()

test_df = major_location_forecasting_df[
    major_location_forecasting_df["ATTENDANCE_DATE"] >= split_date
].copy()

print("Train date range:", train_df["ATTENDANCE_DATE"].min(), "to", train_df["ATTENDANCE_DATE"].max())
print("Test date range:", test_df["ATTENDANCE_DATE"].min(), "to", test_df["ATTENDANCE_DATE"].max())

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
train_location_weekday_baseline = (
    train_df
    .groupby(["LOCATION", "weekday_name"])["location_daily_campus_footfall"]
    .mean()
    .reset_index(name="baseline_expected_footfall")
)

train_location_weekday_baseline.head()

In [ ]:
test_baseline_df = test_df.merge(
    train_location_weekday_baseline,
    on=["LOCATION", "weekday_name"],
    how="left"
)

test_baseline_df["forecast_error"] = (
    test_baseline_df["location_daily_campus_footfall"] -
    test_baseline_df["baseline_expected_footfall"]
)

test_baseline_df["absolute_error"] = (
    test_baseline_df["forecast_error"].abs()
)

test_baseline_df.head()

In [ ]:
test_baseline_evaluation = (
    test_baseline_df
    .groupby("LOCATION")
    .agg(
        average_actual_footfall=("location_daily_campus_footfall", "mean"),
        average_predicted_footfall=("baseline_expected_footfall", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        max_absolute_error=("absolute_error", "max")
    )
    .reset_index()
)

test_baseline_evaluation = test_baseline_evaluation.sort_values(
    "average_actual_footfall",
    ascending=False
)

test_baseline_evaluation

## Time-Based Baseline Forecast Evaluation

A time-based train-test split was used to evaluate the baseline footfall forecast.

Training period:
2025-04-01 to 2025-05-17

Testing period:
2025-05-18 to 2025-05-31

The baseline forecast was created using the historical average footfall for each LOCATION and weekday combination from the training period. This baseline was then applied to the test period.

Key observations:

- The baseline captures broad location-wise and weekday-wise patterns.
- Larger locations naturally show higher absolute forecast errors because their footfall volumes are higher.
- Bengaluru had the highest average actual footfall in the test period and also the highest mean absolute error.
- Hyderabad, Pune, Chennai, Kolkata, and Mumbai show reasonable baseline performance for a simple explainable model.
- Some maximum errors are still high, suggesting that weekday and location alone do not fully explain daily footfall variation.

This confirms that location and weekday are strong forecasting features, but future models may need additional features such as previous-day footfall, previous-week same-day footfall, rolling averages, and holiday or special-day indicators.

In [ ]:
test_baseline_evaluation["mae_pct_of_avg_actual"] = (
    test_baseline_evaluation["mean_absolute_error"] /
    test_baseline_evaluation["average_actual_footfall"] * 100
)

test_baseline_evaluation[
    [
        "LOCATION",
        "average_actual_footfall",
        "average_predicted_footfall",
        "mean_absolute_error",
        "mae_pct_of_avg_actual",
        "max_absolute_error"
    ]
].sort_values("mae_pct_of_avg_actual")

## Time-Based Baseline Forecast Evaluation Insight

A time-based train-test evaluation was performed to test whether historical location-weekday patterns can estimate future campus footfall.

The training period used data from 2025-04-01 to 2025-05-17, and the test period used data from 2025-05-18 to 2025-05-31.

The baseline forecast uses the historical average footfall for each LOCATION and weekday combination.

Key observations:

- Hyderabad shows the lowest relative baseline error among major locations, with MAE around 4.82% of average actual footfall.
- Chennai, Greater Noida, Pune, Gurugram, Kolkata, Kochi, and Coimbatore also show reasonable relative errors.
- Mumbai and Bengaluru show higher relative errors, around 10.37% and 10.53% respectively.
- Larger locations naturally show higher absolute errors due to higher footfall volume.
- The baseline captures broad weekday and location effects, but does not capture special dates, holiday-like dips, previous-day effects, or sudden changes in attendance behavior.

Overall, the location-weekday baseline provides a simple and explainable first forecasting benchmark. Future models can improve this baseline by adding lag variables, rolling averages, holiday indicators, and other time-based features.

In [ ]:
import matplotlib.pyplot as plt

baseline_plot_df = test_baseline_evaluation.sort_values(
    "mae_pct_of_avg_actual",
    ascending=True
)

plt.figure(figsize=(12, 6))

bars = plt.barh(
    baseline_plot_df["LOCATION"],
    baseline_plot_df["mae_pct_of_avg_actual"]
)

plt.title("Baseline Forecast Error by Location")
plt.xlabel("Mean Absolute Error as % of Average Actual Footfall")
plt.ylabel("Location")

plt.gca().invert_yaxis()

for bar in bars:
    width = bar.get_width()
    plt.text(
        width,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}%",
        va="center"
    )

plt.tight_layout()
plt.show()

## Baseline Forecast Error Insight

The baseline forecast error chart shows that the simple location-weekday average model performs reasonably well for several major locations.

Hyderabad and Chennai show the lowest relative errors, with MAE around 4.8% and 5.2% of average actual footfall respectively. Pune and Greater Noida also show reasonable performance with errors around 6.5% to 6.7%.

Mumbai and Bengaluru show the highest relative errors, around 10.4% and 10.5%. This suggests that weekday and location alone may not fully explain daily footfall variation for these larger or more variable locations.

Overall, the baseline provides a useful and explainable first forecasting benchmark. However, future models may improve accuracy by adding additional features such as previous-day footfall, previous-week same-day footfall, rolling averages, holiday/special-day flags, and location-specific patterns.
`

In [ ]:
major_location_forecasting_df = major_location_forecasting_df.sort_values(
    ["LOCATION", "ATTENDANCE_DATE"]
).copy()

major_location_forecasting_df["lag_7_footfall"] = (
    major_location_forecasting_df
    .groupby("LOCATION")["location_daily_campus_footfall"]
    .shift(7)
)

major_location_forecasting_df.head(15)

In [ ]:
split_date = pd.to_datetime("2025-05-18")

train_df = major_location_forecasting_df[
    major_location_forecasting_df["ATTENDANCE_DATE"] < split_date
].copy()

test_df = major_location_forecasting_df[
    major_location_forecasting_df["ATTENDANCE_DATE"] >= split_date
].copy()

In [ ]:
lag7_test_df = test_df.dropna(subset=["lag_7_footfall"]).copy()

lag7_test_df["forecast_error"] = (
    lag7_test_df["location_daily_campus_footfall"] -
    lag7_test_df["lag_7_footfall"]
)

lag7_test_df["absolute_error"] = (
    lag7_test_df["forecast_error"].abs()
)

lag7_evaluation = (
    lag7_test_df
    .groupby("LOCATION")
    .agg(
        average_actual_footfall=("location_daily_campus_footfall", "mean"),
        average_predicted_footfall=("lag_7_footfall", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        max_absolute_error=("absolute_error", "max")
    )
    .reset_index()
)

lag7_evaluation["mae_pct_of_avg_actual"] = (
    lag7_evaluation["mean_absolute_error"] /
    lag7_evaluation["average_actual_footfall"] * 100
)

lag7_evaluation = lag7_evaluation.sort_values(
    "mae_pct_of_avg_actual"
)

lag7_evaluation

## Improved Baseline: Previous Week Same-Day Footfall

An improved baseline was tested using the previous week same-day footfall.

This approach predicts a location’s footfall using the same location’s footfall from seven days earlier.

For example, Bengaluru Tuesday footfall is predicted using Bengaluru’s previous Tuesday footfall.

This baseline captures recent weekly attendance behavior and is useful when hybrid work patterns repeat week over week.

### Key Findings

The previous week same-day baseline improves forecast performance for several major locations compared with the location-weekday average baseline.

Examples:

- Hyderabad improved from approximately 4.82% error to 2.68% error.
- Bengaluru improved from approximately 10.53% error to 5.90% error.
- Chennai improved from approximately 5.20% error to 4.39% error.
- Pune improved from approximately 6.67% error to 5.71% error.

This suggests that recent same-weekday footfall is a strong predictive signal for office attendance forecasting.

However, some locations such as Coimbatore and Mumbai still show relatively higher errors, indicating that additional features may be required for better accuracy.

Possible future features include rolling averages, holiday indicators, special-day flags, previous-day footfall, and location-specific attendance behavior.

In [ ]:
weekday_avg_eval = test_baseline_evaluation[
    [
        "LOCATION",
        "mae_pct_of_avg_actual"
    ]
].copy()

weekday_avg_eval = weekday_avg_eval.rename(
    columns={
        "mae_pct_of_avg_actual": "weekday_average_baseline_error_pct"
    }
)

lag7_eval = lag7_evaluation[
    [
        "LOCATION",
        "mae_pct_of_avg_actual"
    ]
].copy()

lag7_eval = lag7_eval.rename(
    columns={
        "mae_pct_of_avg_actual": "lag7_baseline_error_pct"
    }
)

baseline_model_comparison = weekday_avg_eval.merge(
    lag7_eval,
    on="LOCATION",
    how="inner"
)

baseline_model_comparison["error_improvement_pct_points"] = (
    baseline_model_comparison["weekday_average_baseline_error_pct"] -
    baseline_model_comparison["lag7_baseline_error_pct"]
)

baseline_model_comparison = baseline_model_comparison.sort_values(
    "error_improvement_pct_points",
    ascending=False
)

baseline_model_comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

comparison_df = baseline_model_comparison.sort_values(
    "weekday_average_baseline_error_pct",
    ascending=True
).copy()

x = np.arange(len(comparison_df))
width = 0.35

plt.figure(figsize=(14, 6))

plt.bar(
    x - width/2,
    comparison_df["weekday_average_baseline_error_pct"],
    width,
    label="Location-Weekday Average Baseline"
)

plt.bar(
    x + width/2,
    comparison_df["lag7_baseline_error_pct"],
    width,
    label="Previous Week Same-Day Baseline"
)

plt.xticks(
    x,
    comparison_df["LOCATION"],
    rotation=45,
    ha="right"
)

plt.ylabel("MAE as % of Average Actual Footfall")
plt.title("Baseline Forecast Model Comparison by Location")
plt.legend()

plt.tight_layout()
plt.show()

## Baseline Model Comparison Insight

Two explainable baseline forecasting approaches were compared:

1. Location-Weekday Average Baseline  
   This predicts footfall using the historical average footfall for the same location and weekday.

2. Previous Week Same-Day Baseline  
   This predicts footfall using the same location’s footfall from seven days earlier.

The comparison shows that the Previous Week Same-Day Baseline performs better for most major locations.

Key observations:

- Bengaluru improves significantly, with relative error reducing from approximately 10.5% to 5.9%.
- Hyderabad improves from approximately 4.8% to 2.7%.
- Greater Noida, Chennai, Pune, Gurugram, Kolkata, Kochi, and Mumbai also show improved or competitive performance using the previous-week same-day approach.
- Coimbatore is an exception, where the previous-week same-day baseline performs worse than the location-weekday average baseline.

Overall, this suggests that recent same-weekday footfall is a strong predictive signal for office attendance forecasting. However, some locations may require additional features or location-specific modeling to improve accuracy.

## Improved Explainable Forecast: Weighted Baseline

The previous week same-day baseline uses the same location’s footfall from seven days earlier. While this captures recent weekly attendance behavior, it may be sensitive to unusual spikes or dips from the previous week.

To make the forecast more balanced, this section creates a weighted baseline forecast.

The weighted forecast combines:

- 70% previous week same-day footfall
- 30% historical location-weekday average footfall

This approach uses both recent attendance behavior and historical weekday-location pattern.

This is still an explainable baseline forecasting method, not a production ML model.

In [ ]:
# Use the same train-test split
split_date = pd.to_datetime("2025-05-18")

train_df = major_location_forecasting_df[
    major_location_forecasting_df["ATTENDANCE_DATE"] < split_date
].copy()

test_df = major_location_forecasting_df[
    major_location_forecasting_df["ATTENDANCE_DATE"] >= split_date
].copy()

# Location-weekday baseline from train data
train_location_weekday_baseline = (
    train_df
    .groupby(["LOCATION", "weekday_name"])["location_daily_campus_footfall"]
    .mean()
    .reset_index(name="location_weekday_avg_footfall")
)

# Merge location-weekday average into test data
weighted_test_df = test_df.merge(
    train_location_weekday_baseline,
    on=["LOCATION", "weekday_name"],
    how="left"
)

weighted_test_df.head()

In [ ]:
weighted_test_df["weighted_expected_footfall"] = (
    0.7 * weighted_test_df["lag_7_footfall"] +
    0.3 * weighted_test_df["location_weekday_avg_footfall"]
)

weighted_test_df["forecast_error"] = (
    weighted_test_df["location_daily_campus_footfall"] -
    weighted_test_df["weighted_expected_footfall"]
)

weighted_test_df["absolute_error"] = (
    weighted_test_df["forecast_error"].abs()
)

weighted_test_df.head()

In [ ]:
weighted_evaluation = (
    weighted_test_df
    .groupby("LOCATION")
    .agg(
        average_actual_footfall=("location_daily_campus_footfall", "mean"),
        average_predicted_footfall=("weighted_expected_footfall", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        max_absolute_error=("absolute_error", "max")
    )
    .reset_index()
)

weighted_evaluation["mae_pct_of_avg_actual"] = (
    weighted_evaluation["mean_absolute_error"] /
    weighted_evaluation["average_actual_footfall"] * 100
)

weighted_evaluation = weighted_evaluation.sort_values(
    "mae_pct_of_avg_actual"
)

weighted_evaluation

In [ ]:
weekday_avg_eval = test_baseline_evaluation[
    ["LOCATION", "mae_pct_of_avg_actual"]
].copy()

weekday_avg_eval = weekday_avg_eval.rename(
    columns={
        "mae_pct_of_avg_actual": "location_weekday_avg_error_pct"
    }
)

lag7_eval = lag7_evaluation[
    ["LOCATION", "mae_pct_of_avg_actual"]
].copy()

lag7_eval = lag7_eval.rename(
    columns={
        "mae_pct_of_avg_actual": "lag7_error_pct"
    }
)

weighted_eval = weighted_evaluation[
    ["LOCATION", "mae_pct_of_avg_actual"]
].copy()

weighted_eval = weighted_eval.rename(
    columns={
        "mae_pct_of_avg_actual": "weighted_baseline_error_pct"
    }
)

model_comparison = (
    weekday_avg_eval
    .merge(lag7_eval, on="LOCATION", how="inner")
    .merge(weighted_eval, on="LOCATION", how="inner")
)

model_comparison = model_comparison.sort_values(
    "weighted_baseline_error_pct"
)

model_comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

comparison_df = model_comparison.copy()

x = np.arange(len(comparison_df))
width = 0.25

plt.figure(figsize=(15, 6))

plt.bar(
    x - width,
    comparison_df["location_weekday_avg_error_pct"],
    width,
    label="Location-Weekday Average"
)

plt.bar(
    x,
    comparison_df["lag7_error_pct"],
    width,
    label="Previous Week Same-Day"
)

plt.bar(
    x + width,
    comparison_df["weighted_baseline_error_pct"],
    width,
    label="Weighted Baseline"
)

plt.xticks(
    x,
    comparison_df["LOCATION"],
    rotation=45,
    ha="right"
)

plt.ylabel("MAE as % of Average Actual Footfall")
plt.title("Forecast Baseline Model Comparison by Location")
plt.legend()

plt.tight_layout()
plt.show()

## Three-Baseline Comparison Insight

Three explainable forecasting baselines were compared:

1. Location-Weekday Average Baseline
2. Previous Week Same-Day Baseline
3. Weighted Baseline combining previous week same-day footfall and location-weekday average

The weighted baseline improves over the simple location-weekday average baseline for most locations, showing that combining recent and historical patterns improves stability.

However, the best-performing method differs by location:

- Previous Week Same-Day performs best for Hyderabad, Bengaluru, and Mumbai.
- Weighted Baseline performs best for Greater Noida, Chennai, Gurugram, Pune, Kochi, and Kolkata.
- Location-Weekday Average performs best for Coimbatore.

This suggests that a location-specific forecasting strategy may be more appropriate than applying one single forecasting method to all locations.

In [ ]:
champion_model_df = model_comparison.copy()

def select_best_model(row):
    errors = {
        "Location-Weekday Average Baseline": row["location_weekday_avg_error_pct"],
        "Previous Week Same-Day Baseline": row["lag7_error_pct"],
        "Weighted Baseline": row["weighted_baseline_error_pct"]
    }
    
    best_model = min(errors, key=errors.get)
    best_error = errors[best_model]
    
    return pd.Series({
        "best_model": best_model,
        "best_model_error_pct": best_error
    })

champion_model_df[["best_model", "best_model_error_pct"]] = (
    champion_model_df.apply(select_best_model, axis=1)
)

champion_model_df = champion_model_df[
    [
        "LOCATION",
        "location_weekday_avg_error_pct",
        "lag7_error_pct",
        "weighted_baseline_error_pct",
        "best_model",
        "best_model_error_pct"
    ]
].sort_values("best_model_error_pct")

champion_model_df

## Champion Baseline Model by Location

The comparison of three explainable forecasting baselines shows that no single method performs best for every location.

The best-performing method varies by location:

- Previous Week Same-Day Baseline performs best for Hyderabad, Bengaluru, and Mumbai.
- Weighted Baseline performs best for Greater Noida, Chennai, Gurugram, Pune, Kochi, and Kolkata.
- Location-Weekday Average Baseline performs best for Coimbatore.

This suggests that a location-specific champion baseline approach is more suitable than applying one common forecasting method across all locations.

The final forecasting prototype can therefore choose the best-performing baseline model for each location based on validation results.

In [ ]:
# Create final location-weekday baseline using full available historical data
final_location_weekday_baseline = (
    major_location_forecasting_df
    .groupby(["LOCATION", "weekday_name"])["location_daily_campus_footfall"]
    .mean()
    .reset_index(name="location_weekday_avg_footfall")
)

# Create champion model lookup
champion_model_lookup = champion_model_df[
    ["LOCATION", "best_model"]
].copy()


def predict_expected_footfall_champion(input_date, input_location):
    """
    Predict expected campus footfall for a given date and location
    using the best-performing baseline model selected for that location.

    Champion model options:
    1. Previous Week Same-Day Baseline
    2. Weighted Baseline
    3. Location-Weekday Average Baseline
    """

    input_date = pd.to_datetime(input_date)
    input_location = str(input_location).upper().strip()

    weekday_name = input_date.day_name()
    lag_7_date = input_date - pd.Timedelta(days=7)

    # Get location champion model
    champion_match = champion_model_lookup[
        champion_model_lookup["LOCATION"] == input_location
    ]

    if champion_match.empty:
        return {
            "forecast_date": input_date.date(),
            "location": input_location,
            "weekday": weekday_name,
            "expected_footfall": None,
            "method_used": "No champion model available for this location"
        }

    best_model = champion_match["best_model"].iloc[0]

    # Get lag-7 actual footfall if available
    lag_match = major_location_forecasting_df[
        (major_location_forecasting_df["ATTENDANCE_DATE"] == lag_7_date) &
        (major_location_forecasting_df["LOCATION"] == input_location)
    ]

    lag_7_footfall = None

    if not lag_match.empty:
        lag_7_footfall = lag_match["location_daily_campus_footfall"].iloc[0]

    # Get historical location-weekday average
    avg_match = final_location_weekday_baseline[
        (final_location_weekday_baseline["LOCATION"] == input_location) &
        (final_location_weekday_baseline["weekday_name"] == weekday_name)
    ]

    location_weekday_avg = None

    if not avg_match.empty:
        location_weekday_avg = avg_match["location_weekday_avg_footfall"].iloc[0]

    # Apply champion model logic
    if best_model == "Previous Week Same-Day Baseline":
        if lag_7_footfall is not None:
            prediction = lag_7_footfall
            method_used = "Previous Week Same-Day Baseline"
        else:
            prediction = location_weekday_avg
            method_used = "Fallback: Location-Weekday Average Baseline"

    elif best_model == "Weighted Baseline":
        if lag_7_footfall is not None and location_weekday_avg is not None:
            prediction = (0.7 * lag_7_footfall) + (0.3 * location_weekday_avg)
            method_used = "Weighted Baseline"
        elif location_weekday_avg is not None:
            prediction = location_weekday_avg
            method_used = "Fallback: Location-Weekday Average Baseline"
        else:
            prediction = None
            method_used = "No baseline available"

    elif best_model == "Location-Weekday Average Baseline":
        prediction = location_weekday_avg
        method_used = "Location-Weekday Average Baseline"

    else:
        prediction = None
        method_used = "Unknown model"

    return {
        "forecast_date": input_date.date(),
        "location": input_location,
        "weekday": weekday_name,
        "expected_footfall": round(prediction) if prediction is not None else None,
        "method_used": method_used,
        "champion_model": best_model
    }

In [ ]:
test_predictions = pd.DataFrame([
    predict_expected_footfall_champion("2025-06-02", "BENGALURU"),
    predict_expected_footfall_champion("2025-06-03", "HYDERABAD"),
    predict_expected_footfall_champion("2025-06-04", "PUNE"),
    predict_expected_footfall_champion("2025-06-05", "MUMBAI"),
    predict_expected_footfall_champion("2025-06-06", "COIMBATORE")
])

test_predictions

## Manager-facing Champion Forecast Function Output

The champion forecast function successfully generates expected footfall for a given date and location.

Instead of applying one fixed method across all locations, the function uses the best-performing baseline model selected during validation for each location.

Example outputs:

- Bengaluru uses the Previous Week Same-Day Baseline.
- Pune uses the Weighted Baseline.
- Coimbatore uses the Location-Weekday Average Baseline.

This makes the forecasting prototype more flexible and location-specific.

The output includes:

- Forecast date
- Location
- Weekday
- Expected footfall
- Method used
- Champion model selected for the location

This creates a manager-facing forecasting output that can be used to estimate expected campus footfall for selected future dates and locations.

In [ ]:
future_dates = pd.date_range(
    start=daily_footfall["ATTENDANCE_DATE"].max() + pd.Timedelta(days=1),
    periods=7,
    freq="D"
)

locations_to_forecast = major_forecast_locations

forecast_results = []

for forecast_date in future_dates:
    for location in locations_to_forecast:
        forecast_results.append(
            predict_expected_footfall_champion(
                forecast_date,
                location
            )
        )

forecast_output = pd.DataFrame(forecast_results)

forecast_output.head(20)

In [ ]:
daily_total_forecast = (
    forecast_output
    .groupby(["forecast_date", "weekday"])["expected_footfall"]
    .sum()
    .reset_index(name="total_expected_footfall")
)

daily_total_forecast


In [ ]:
location_forecast_summary = (
    forecast_output
    .groupby("location")["expected_footfall"]
    .agg(
        average_expected_footfall="mean",
        max_expected_footfall="max",
        min_expected_footfall="min"
    )
    .reset_index()
    .sort_values("average_expected_footfall", ascending=False)
)

location_forecast_summary

In [ ]:
method_summary = (
    forecast_output["method_used"]
    .value_counts()
    .reset_index()
)

method_summary.columns = [
    "method_used",
    "forecast_row_count"
]

method_summary["forecast_row_pct"] = (
    method_summary["forecast_row_count"] /
    method_summary["forecast_row_count"].sum() * 100
)

method_summary

In [ ]:
output_file = "office_footfall_forecast_output.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    forecast_output.to_excel(
        writer,
        sheet_name="Forecast_Detail",
        index=False
    )
    
    daily_total_forecast.to_excel(
        writer,
        sheet_name="Daily_Total_Forecast",
        index=False
    )
    
    location_forecast_summary.to_excel(
        writer,
        sheet_name="Location_Summary",
        index=False
    )
    
    method_summary.to_excel(
        writer,
        sheet_name="Method_Summary",
        index=False
    )

print(f"Forecast workbook saved successfully: {output_file}")

## Manager-facing Forecast Workbook

The final forecast output was exported as an Excel workbook to make the result easier to review and use.

The workbook contains four sheets:

1. Forecast_Detail  
   Date-location level expected footfall forecast.

2. Daily_Total_Forecast  
   Total expected footfall by forecast date across selected major locations.

3. Location_Summary  
   Average, maximum, and minimum expected footfall by location.

4. Method_Summary  
   Summary of forecasting methods used in the output.

This workbook is intended to provide a short-term planning view for location-wise office occupancy.

## Forecasting Model Card

**Prediction Target:**  
Location-wise daily campus footfall

**Prediction Unit:**  
Forecast date + Location

**Forecast Horizon:**  
Next 7 calendar days

**Current Approach:**  
Location-specific champion baseline model

**Baseline Methods Tested:**

1. Location-Weekday Average Baseline  
2. Previous Week Same-Day Baseline  
3. Weighted Baseline

**Evaluation Method:**  
Time-based train-test split

**Training Period:**  
2025-04-01 to 2025-05-17

**Testing Period:**  
2025-05-18 to 2025-05-31

**Best Use Case:**  
Short-term location-wise office footfall estimation and occupancy planning

**Current Limitations:**

- Only 61 days of historical data are available.
- Holiday and special-event calendars are not included.
- Project/team-level schedules are not included.
- The current approach is an explainable baseline forecasting prototype, not a production ML model.

**Future Improvements:**

- Add more historical months.
- Add holiday and special-day flags.
- Add previous-day and rolling-average features.
- Test ML regression models after sufficient historical data is available.
- Export outputs to Excel or Power BI for business consumption.

## Key Conclusions

1. Office footfall can be defined reliably using unique employee counts from Campus / Office Presence records.

2. The available data contains 61 continuous calendar dates from 2025-04-01 to 2025-05-31 with no missing dates.

3. Campus footfall shows a clear weekday pattern, with Tuesday and Wednesday generally higher and weekends much lower.

4. Employee weekly office-day combinations show that 3-day campus attendance is clustered around specific weekday blocks, especially Monday-Tuesday-Wednesday and Tuesday-Wednesday-Thursday.

5. Location-wise footfall is concentrated in major locations such as Bengaluru, Hyderabad, Pune, Chennai, Kolkata, and Mumbai.

6. Three explainable forecasting baselines were tested using a time-based train-test split.

7. The best-performing forecasting method differs by location, so a location-specific champion baseline approach was used.

8. The final prototype can generate expected footfall by future date and location and export results into a manager-facing Excel workbook.

9. The current model should be treated as an explainable forecasting prototype and can be improved further with more historical data and additional business context.

## Day-wise Forecast Summary

Along with location-wise expected footfall, a day-wise forecast summary was created.

This view shows the total expected footfall for each forecast date across the selected major locations.

This helps understand overall expected office occupancy by day and can support planning for facilities, cafeteria, transport, security, and seating.

In [ ]:
daily_total_forecast = (
    forecast_output
    .groupby(["forecast_date", "weekday"])["expected_footfall"]
    .sum()
    .reset_index(name="total_expected_footfall")
)

daily_total_forecast

## Day-wise Forecast Insight

The day-wise forecast summary shows expected total footfall across the selected major locations for each forecast date.

The forecast follows a clear weekly pattern:

- Wednesday has the highest expected footfall at around 56,841.
- Tuesday is also high at around 51,725.
- Monday, Thursday, and Friday show moderate expected footfall.
- Saturday and Sunday show very low expected footfall compared with regular weekdays.

This aligns with the historical weekday pattern observed earlier, where Tuesday and Wednesday had higher campus attendance and weekends had much lower attendance.

This day-wise forecast view makes the output easier to use for short-term occupancy and facility planning.

## Month-on-Month Footfall Comparison: April vs May

This section compares campus footfall between April 2025 and May 2025.

The goal is to understand whether overall office footfall increased, decreased, or remained stable from one month to the next.

The comparison is first done at overall daily footfall level.

In [ ]:
daily_footfall["month_name"] = daily_footfall["ATTENDANCE_DATE"].dt.month_name()

monthly_footfall_summary = (
    daily_footfall
    .groupby("month_name")["daily_campus_footfall"]
    .agg(
        average_daily_footfall="mean",
        median_daily_footfall="median",
        max_daily_footfall="max",
        min_daily_footfall="min",
        number_of_days="count"
    )
    .reset_index()
)

monthly_footfall_summary

In [ ]:
april_avg = monthly_footfall_summary.loc[
    monthly_footfall_summary["month_name"] == "April",
    "average_daily_footfall"
].iloc[0]

may_avg = monthly_footfall_summary.loc[
    monthly_footfall_summary["month_name"] == "May",
    "average_daily_footfall"
].iloc[0]

overall_month_comparison = pd.DataFrame({
    "metric": ["Average Daily Footfall"],
    "april_value": [april_avg],
    "may_value": [may_avg],
    "difference_may_minus_april": [may_avg - april_avg],
    "pct_change": [((may_avg - april_avg) / april_avg) * 100]
})

overall_month_comparison

## Overall April vs May Footfall Insight

The overall month-on-month comparison shows that average daily campus footfall decreased from April to May.

- April average daily footfall: approximately 35,248 employees
- May average daily footfall: approximately 32,719 employees
- Difference: approximately -2,530 employees per day
- Percentage change: approximately -7.18%

This indicates that May had lower average daily campus footfall compared with April.

This comparison provides a high-level month-on-month view and helps understand whether overall office attendance increased or decreased between the two months.
`

## Location-wise April vs May Footfall Comparison

After comparing overall footfall between April and May, this section compares month-on-month footfall at location level.

The objective is to identify which major locations showed an increase or decrease in average daily campus footfall from April to May.

In [ ]:
location_daily_footfall["ATTENDANCE_DATE"] = pd.to_datetime(
    location_daily_footfall["ATTENDANCE_DATE"]
)

location_daily_footfall["month_name"] = (
    location_daily_footfall["ATTENDANCE_DATE"]
    .dt.month_name()
)

In [ ]:
location_monthly_footfall = (
    location_daily_footfall
    .groupby(["LOCATION", "month_name"])["location_daily_campus_footfall"]
    .mean()
    .reset_index(name="average_daily_footfall")
)

location_monthly_footfall.head()

In [ ]:
location_monthly_comparison = (
    location_monthly_footfall
    .pivot_table(
        index="LOCATION",
        columns="month_name",
        values="average_daily_footfall",
        fill_value=0
    )
    .reset_index()
)

location_monthly_comparison.head()

In [ ]:
location_monthly_comparison["difference_may_minus_april"] = (
    location_monthly_comparison["May"] -
    location_monthly_comparison["April"]
)

location_monthly_comparison["pct_change"] = (
    location_monthly_comparison["difference_may_minus_april"] /
    location_monthly_comparison["April"] * 100
)

location_monthly_comparison = location_monthly_comparison.sort_values(
    "difference_may_minus_april"
)

location_monthly_comparison.head(15)

In [ ]:
major_location_monthly_comparison = location_monthly_comparison[
    location_monthly_comparison["LOCATION"].isin(major_forecast_locations)
].copy()

major_location_monthly_comparison = major_location_monthly_comparison.sort_values(
    "difference_may_minus_april"
)

major_location_monthly_comparison

## Location-wise April vs May Footfall Insight

The location-wise month-on-month comparison shows that average daily campus footfall decreased from April to May for most major locations.

Key observations:

- Hyderabad showed the largest absolute decrease, with average daily footfall reducing by approximately 715 employees.
- Pune also showed a significant decrease of approximately 508 employees per day.
- Bengaluru decreased by approximately 350 employees per day.
- Mumbai and Chennai also showed meaningful decreases.
- Kolkata showed a relatively smaller decrease compared with other major locations.
- Kochi was the only major location in this group that showed a small increase from April to May.

Overall, May had lower average daily campus footfall than April across most major locations. This month-on-month comparison provides additional context for interpreting forecast outputs and future attendance trends.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_df = major_location_monthly_comparison.copy()

# Sort by April average footfall for cleaner visual
plot_df = plot_df.sort_values("April", ascending=False)

x = np.arange(len(plot_df))
width = 0.35

plt.figure(figsize=(14, 6))

plt.bar(
    x - width/2,
    plot_df["April"],
    width,
    label="April Average Footfall"
)

plt.bar(
    x + width/2,
    plot_df["May"],
    width,
    label="May Average Footfall"
)

plt.xticks(
    x,
    plot_df["LOCATION"],
    rotation=45,
    ha="right"
)

plt.title("Location-wise Average Daily Footfall: April vs May")
plt.xlabel("Location")
plt.ylabel("Average Daily Footfall")
plt.legend()

plt.tight_layout()
plt.show()

## Updated Manager-facing Forecast Workbook

Based on feedback, the forecast workbook was updated to include both forecast output and previous month comparison.

The workbook contains:

1. Forecast_Detail  
   Date-location level expected footfall forecast.

2. Daily_Total_Forecast  
   Day-wise expected total footfall across selected major locations.

3. Location_Summary  
   Average, maximum, and minimum expected footfall by location for the forecast period.

4. Method_Summary  
   Summary of forecasting methods used.

5. Overall_Monthly_Comparison  
   Overall April vs May average daily footfall comparison.

6. Location_Monthly_Comparison  
   Location-wise April vs May average daily footfall difference and percentage change.

This provides both a forward-looking forecast view and a historical month-on-month comparison.

In [ ]:
forecast_start_date = pd.Timestamp.today().normalize()

forecast_end_date = forecast_start_date + pd.offsets.MonthEnd(0)

future_dates_month_end = pd.date_range(
    start=forecast_start_date,
    end=forecast_end_date,
    freq="D"
)

future_dates_month_end

In [ ]:
forecast_start_date = pd.to_datetime("2026-07-15")
forecast_end_date = forecast_start_date + pd.offsets.MonthEnd(0)

future_dates_month_end = pd.date_range(
    start=forecast_start_date,
    end=forecast_end_date,
    freq="D"
)

future_dates_month_end

In [ ]:
final_location_weekday_baseline = (
    major_location_forecasting_df
    .groupby(["LOCATION", "weekday_name"])["location_daily_campus_footfall"]
    .mean()
    .reset_index(name="expected_footfall")
)

final_location_weekday_baseline.head()

In [ ]:
forecast_results_month_end = []

for forecast_date in future_dates_month_end:
    weekday_name = forecast_date.day_name()
    
    for location in major_forecast_locations:
        match = final_location_weekday_baseline[
            (final_location_weekday_baseline["LOCATION"] == location) &
            (final_location_weekday_baseline["weekday_name"] == weekday_name)
        ]
        
        if not match.empty:
            expected_footfall = round(match["expected_footfall"].iloc[0])
        else:
            expected_footfall = None
        
        forecast_results_month_end.append({
            "forecast_date": forecast_date.date(),
            "forecast_week": forecast_date.isocalendar().week,
            "week_start_date": (forecast_date - pd.Timedelta(days=forecast_date.weekday())).date(),
            "weekday": weekday_name,
            "location": location,
            "expected_footfall": expected_footfall,
            "method_used": "Location-Weekday Average Baseline"
        })

forecast_output_month_end = pd.DataFrame(forecast_results_month_end)

forecast_output_month_end.head(20)

In [ ]:
forecast_output_month_end["week_label"] = (
    "Week " + forecast_output_month_end["forecast_week"].astype(str)
)

forecast_output_month_end.head()

In [ ]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

forecast_output_month_end["weekday"] = pd.Categorical(
    forecast_output_month_end["weekday"],
    categories=weekday_order,
    ordered=True
)

presentable_week_location_pivot = (
    forecast_output_month_end
    .pivot_table(
        index=["week_label", "location"],
        columns="weekday",
        values="expected_footfall",
        aggfunc="sum",
        fill_value=0
    )
)

presentable_week_location_pivot = presentable_week_location_pivot.reindex(
    columns=weekday_order,
    fill_value=0
)

presentable_week_location_pivot["Grand Total"] = (
    presentable_week_location_pivot.sum(axis=1)
)

presentable_week_location_pivot

In [ ]:
presentable_location_weekday_pivot = (
    forecast_output_month_end
    .pivot_table(
        index="location",
        columns="weekday",
        values="expected_footfall",
        aggfunc="sum",
        fill_value=0
    )
)

presentable_location_weekday_pivot = presentable_location_weekday_pivot.reindex(
    columns=weekday_order,
    fill_value=0
)

presentable_location_weekday_pivot["Grand Total"] = (
    presentable_location_weekday_pivot.sum(axis=1)
)

grand_total_row = pd.DataFrame(
    presentable_location_weekday_pivot.sum(axis=0)
).T

grand_total_row.index = ["Grand Total"]

presentable_location_weekday_pivot_with_total = pd.concat(
    [presentable_location_weekday_pivot, grand_total_row]
)

presentable_location_weekday_pivot_with_total

In [ ]:
daily_total_forecast_month_end = (
    forecast_output_month_end
    .groupby(["forecast_date", "week_label", "weekday"])["expected_footfall"]
    .sum()
    .reset_index(name="total_expected_footfall")
)

daily_total_forecast_month_end

In [ ]:
output_file = "office_footfall_forecast_till_month_end.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    presentable_location_weekday_pivot_with_total.to_excel(
        writer,
        sheet_name="Location_Weekday_Pivot"
    )
    
    presentable_week_location_pivot.to_excel(
        writer,
        sheet_name="Week_Location_Pivot"
    )
    
    forecast_output_month_end.to_excel(
        writer,
        sheet_name="Forecast_Detail",
        index=False
    )
    
    daily_total_forecast_month_end.to_excel(
        writer,
        sheet_name="Daily_Total_Forecast",
        index=False
    )
    
    overall_month_comparison.to_excel(
        writer,
        sheet_name="Overall_Monthly_Comparison",
        index=False
    )
    
    major_location_monthly_comparison.to_excel(
        writer,
        sheet_name="Location_Monthly_Comparison",
        index=False
    )

print(f"Forecast workbook saved successfully: {output_file}")